# 08: Human review

The agent of notebook 07 writes a report, but it decides nothing. A person approves or
rejects each possible problem, and only then is a listing flagged and a message to the
seller drafted. Nothing is ever sent to a seller: the draft is saved for a person.

This notebook looks at the four parts that make that work:

- **interrupt**: the graph pauses in the middle of an audit and waits for a person;
- **the checkpointer**: the paused audit is saved in Postgres, so it survives a restart;
- **the audit log**: every report and decision is kept, and the database refuses changes;
- **the review API**: FastAPI runs the audits, streams their steps, and takes decisions.

The review screen, a Streamlit app, is the API's client; section 7 says how to run it.
As in notebook 07, a scripted stand-in plays the model, so nothing here costs money.

In [1]:
# Quiet the model loading: no progress bars, no notes about checking for updates.
# This has to happen before the libraries are imported, so it gets a cell of its own.
import os
import warnings

os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore", message="IProgress not found")

In [2]:
import json

import httpx
import psycopg
from huggingface_hub.utils import logging as hub_logging
from IPython.display import Markdown, display
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from mcp.shared.memory import create_connected_server_and_client_session
from psycopg import sql
from pydantic import ValidationError

from listingaudit.agent.brain import Brain
from listingaudit.agent.graph import build_graph
from listingaudit.agent.report import Decision, FindingDraft, KindChoice, ReportDraft, Review
from listingaudit.agent.scripted import ScriptedModel
from listingaudit.agent.toolbox import Toolbox
from listingaudit.config import settings
from listingaudit.market.rules import Rule
from listingaudit.review.api import app
from listingaudit.review.log import open_pool
from listingaudit.review.service import ReviewService
from listingaudit.toolbox import server

hub_logging.set_verbosity_error()
# The photo check stays free here: CLIP only, no vision model
settings.vision_api_key = None
# The notebook keeps its audits in a schema of its own, dropped at the end, so they
# never show up on the review screen
SCHEMA = "notebook_review"


def p0171() -> list:
    """The model's answers for an audit of P0171, as in notebook 07."""

    def draft(messages) -> ReportDraft:
        text = messages[-1].content
        heading = "The rulebook passages found for them:\n"
        passages, _ = json.JSONDecoder().raw_decode(text, text.index(heading) + len(heading))
        finding = FindingDraft(
            rule=Rule.PRIOR_PRICE,
            explanation="The badge says -42%, but the price is only 25.0% below €19.99, the "
            "lowest price in the 30 days before the reduction.",
            suggested_fix="Show -25% on the badge, or leave the badge out.",
            quotes=[passages[0]["chunk_id"]],
        )
        return ReportDraft(summary="One possible problem: the -42% badge.", findings=[finding])

    search = {"name": "search_rulebook", "args": {"query": "Article 6a prior price"}, "id": "1"}
    return [
        KindChoice(kind="textile", reason="A cushion cover is a textile."),
        AIMessage("", tool_calls=[search]),
        AIMessage("The problem has a passage."),
        draft,
    ]

## 1. A graph that pauses

`build_graph` takes a checkpointer, and with one it adds two steps after the report:
**human_review**, where the graph pauses until a person decides, and **draft_message**,
which drafts the message to the seller from the findings the person approved. Without a
checkpointer, as on the command line of notebook 07, the graph ends with the report.

In [3]:
async with create_connected_server_and_client_session(server.mcp) as session:
    toolbox = await Toolbox.connect(session)
    graph = build_graph(toolbox, Brain(ScriptedModel(script=[])), checkpointer=InMemorySaver())
display(Markdown(f"```mermaid\n{graph.get_graph().draw_mermaid()}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	read_listing(read_listing)
	pick_rules(pick_rules)
	run_checks(run_checks)
	look_up_law(look_up_law)
	write_report(write_report)
	self_check(self_check)
	search(search<hr/><small><em>__tools = search_rulebook</em></small>)
	escalate(escalate)
	build_report(build_report)
	human_review(human_review)
	draft_message(draft_message)
	__end__([<p>__end__</p>]):::last
	__start__ --> read_listing;
	build_report --> human_review;
	escalate --> write_report;
	human_review --> draft_message;
	look_up_law -.-> search;
	look_up_law -.-> write_report;
	pick_rules --> run_checks;
	read_listing --> pick_rules;
	run_checks -.-> build_report;
	run_checks -.-> look_up_law;
	search --> look_up_law;
	self_check -.-> build_report;
	self_check -.-> escalate;
	self_check -.-> write_report;
	write_report --> self_check;
	draft_message --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

The pause is one function call. Inside human_review, `interrupt(question)` stops the graph
and hands the question to whoever runs it. The graph's state is saved by the
**checkpointer**, under a **thread id**, which here names one audit. Later, the program
answers with `Command(resume=answer)` on the same thread, and the graph carries on from
where it stopped. A pause can last a second or a week: nothing waits in memory.

Here the checkpointer is `InMemorySaver`, which keeps the state in a Python dictionary.
Each call below builds the graph anew, with the same checkpointer: the paused audit lives
in the checkpointer, not in the graph object.

In [4]:
saver = InMemorySaver()
thread = {"configurable": {"thread_id": "notebook-audit-1"}}


async def on_graph(work, model=None):
    """Build the graph on the saver above, and run work(graph) with it."""
    async with create_connected_server_and_client_session(server.mcp) as session:
        brain = Brain(model or ScriptedModel(script=[]))
        graph = build_graph(await Toolbox.connect(session), brain, checkpointer=saver)
        return await work(graph)


result = await on_graph(
    lambda graph: graph.ainvoke({"offer_id": "P0171"}, thread), ScriptedModel(script=p0171())
)
[question] = result["__interrupt__"]
print(question.value)

{'decide_on': ['prior_price'], 'errors': []}


The graph stopped at human_review and asked its question: decide on the prior_price
finding. The state says where it waits:

In [5]:
snapshot = await on_graph(lambda graph: graph.aget_state(thread))
print("Next step:", snapshot.next)
print("Report summary:", snapshot.values["report"]["summary"])

Next step: ('human_review',)
Report summary: One possible problem: the -42% badge.


## 2. Answering the question

The interrupt also carries the shape of the answer it expects, as a JSON schema made from
the `Review` class. LangGraph checks every answer against it, and the API and the screen
can read it too:

In [6]:
print(json.dumps(question.response_schema["properties"], indent=1)[:700])

{
 "reviewer": {
  "description": "Who decided",
  "minLength": 1,
  "title": "Reviewer",
  "type": "string"
 },
 "decisions": {
  "items": {
   "$ref": "#/$defs/Decision"
  },
  "title": "Decisions",
  "type": "array"
 },
 "note": {
  "default": "",
  "description": "Anything else the reviewer noticed",
  "title": "Note",
  "type": "string"
 }
}


An answer that doesn't fit the schema, such as one without a reviewer's name, is refused
before anything is saved. The audit still waits at the same question:

In [7]:
async def answer_without_a_name(graph):
    try:
        await graph.ainvoke(Command(resume={"reviewer": ""}), thread)
    except ValidationError as error:
        print("Refused:", error.errors()[0]["loc"], error.errors()[0]["msg"])
    print("Next step:", (await graph.aget_state(thread)).next)


await on_graph(answer_without_a_name)

Refused: ('reviewer',) String should have at least 1 character
Next step: ('human_review',)


An answer that fits but leaves a finding open is caught by human_review itself, which
asks again, now with the reason. On every resume the node runs again from its first line,
and each `interrupt()` in turn returns the answer it got: the first one returns this
incomplete review, and the second one pauses.

In [8]:
result = await on_graph(
    lambda graph: graph.ainvoke(Command(resume={"reviewer": "Test Reviewer"}), thread)
)
print(result["__interrupt__"][0].value)

{'decide_on': ['prior_price'], 'errors': ['Approve or reject the finding on Honest discounts (the 30-day rule).']}


A complete review lets the graph go on. draft_message builds the message to the seller in
plain code, from text that is already checked: each explanation passed the self-check,
and the person approved it and its fix. No new model call is needed.

In [9]:
review = Review(
    reviewer="Test Reviewer", decisions=[Decision(rule=Rule.PRIOR_PRICE, approved=True)]
)
answer = Command(resume=review.model_dump(mode="json"))
result = await on_graph(lambda graph: graph.ainvoke(answer, thread))
print("Subject:", result["message"]["subject"])
print()
print(result["message"]["body"])

Subject: Your listing P0171: 1 possible problem to look at

Hello,

We have checked your listing P0171, "Amazon Brand – Stone & Beam Classic Outdoor Patterened Throw Pillow - 20 x 20 Inch, Antique Red", and found 1 possible problem with the EU rules for selling to consumers. Please look at it and change the listing where needed.

1. Honest discounts (the 30-day rule)
The badge says -42%, but the price is only 25.0% below €19.99, the lowest price in the 30 days before the reduction.
What to change: Show -25% on the badge, or leave the badge out.
Where the rule comes from: Directive 98/6/EC Art. 6a(1); Directive 98/6/EC Art. 6a(2); PAngV § 11(1); Commission guidance on Article 6a, section 2.1; Case C-330/23 (Aldi Süd), operative part

If you think we got something wrong, reply to this message and tell us why.

Kind regards,
Test Reviewer


## 3. Surviving a restart

An `InMemorySaver` dies with the process. The review service uses LangGraph's Postgres
checkpointer instead, `AsyncPostgresSaver`, so a stopped or crashed API loses no paused
audit. The service also writes every report and decision to the audit log.

`service_run` opens a service the way the API does: a pool of database connections that
work in the review schema, the toolbox, and the graph with its checkpointer. When it
returns, all of that is closed, as if the API had stopped.

In [10]:
async def service_run(work, model=None):
    """Open a review service, run work(service) with it, and close it again."""
    async with (
        open_pool(SCHEMA) as pool,
        create_connected_server_and_client_session(server.mcp) as session,
    ):
        brain = Brain(model or ScriptedModel(script=[]))
        service = await ReviewService.create(pool, await Toolbox.connect(session), brain)
        return await work(service)


async def audit_p0171(service):
    async for progress in service.audit("P0171"):
        print(f"{progress.step} ({progress.seconds} s): {progress.detail}")
    return progress.audit_id


audit_id = await service_run(audit_p0171, ScriptedModel(script=p0171()))

read_listing (0.1 s): Amazon Brand – Stone & Beam Classic Outdoor Patterened Throw Pillow - 


pick_rules (0.1 s): textile (A cushion cover is a textile.) -> prior_price, manufacturer_info, eu_responsible_person, fibre_composition, photo_matches_text


run_checks (0.2 s): 5 checks, 1 possible problem
look_up_law (0.2 s): Article 6a prior price


search (0.3 s): 5 passages
look_up_law (0.3 s): done searching
write_report (0.3 s): draft 1
self_check (0.3 s): passed
build_report (0.3 s): report ready


waiting (0.3 s): One possible problem: the -42% badge.


That service is closed now. A new one, with new connections and a model that has nothing
to say, finds the audit where it paused:

In [11]:
waiting = await service_run(lambda service: service.waiting())
print([(audit.offer_id, audit.status, str(audit.audit_id) == str(audit_id)) for audit in waiting])

[('P0171', 'waiting', True)]


In the database, the paused audit is one row of `checkpoints`, with the bigger values,
such as the conversation with the model, in `checkpoint_blobs`. The service saves a
checkpoint only when the graph pauses or ends (LangGraph's `durability="exit"`), not after
every step: the pause is what has to survive, and an audit cut off halfway is cheaper to
run again.

In [12]:
with psycopg.connect(settings.database_url) as conn:
    for table in ("checkpoints", "checkpoint_blobs", "checkpoint_writes"):
        query = sql.SQL("SELECT count(*) FROM {}.{} WHERE thread_id = %s").format(
            sql.Identifier(SCHEMA), sql.Identifier(table)
        )
        print(table, conn.execute(query, (str(audit_id),)).fetchone()[0])

checkpoints 1
checkpoint_blobs 7
checkpoint_writes 1


This time the reviewer rejects the finding, with a reason. Nothing is approved, so the
listing is not flagged and there is no message to draft:

In [13]:
rejected = Decision(rule=Rule.PRIOR_PRICE, approved=False, reason="Legal cleared this badge.")
review = Review(reviewer="Test Reviewer", decisions=[rejected])
audit = await service_run(lambda service: service.review(audit_id, review))
print("Status:", audit.status, "| flagged:", audit.flagged, "| message:", audit.message)

Status: reviewed | flagged: False | message: None


## 4. The audit log

The checkpoints are the agent's working memory: binary, internal, and safe to delete once
an audit is done. The audit log is the record: one readable row for each report and each
decision, with who decided and when.

In [14]:
for line in await service_run(lambda service: service.history()):
    print(f"{line.at:%H:%M:%S}  {line.event:<9} {line.detail}")

20:23:36  reviewed  Test Reviewer approved 0 of 1 finding
20:23:36  reported  One possible problem: the -42% badge.


Rows are only ever added. Triggers in `db/review.sql` refuse every change and deletion,
even from the database's owner:

In [15]:
with psycopg.connect(settings.database_url, autocommit=True) as conn:
    change = sql.SQL("UPDATE {}.audit_log SET offer_id = 'P0001'").format(sql.Identifier(SCHEMA))
    try:
        conn.execute(change)
    except psycopg.errors.RaiseException as error:
        print("Refused:", error.diag.message_primary)

Refused: The audit log only takes new rows


## 5. The review API

The API is a FastAPI app with one service, opened when the app starts (its **lifespan**)
and handed to each endpoint by **dependency injection**. These are its endpoints:

In [16]:
for path, methods in app.openapi()["paths"].items():
    for method, operation in methods.items():
        print(f"{method.upper():<5} {path:<32} {operation['summary']}")

GET   /listings/{offer_id}             Read Listing
GET   /listings/{offer_id}/photo       Read Photo
POST  /listings/{offer_id}/audits      Start Audit
GET   /audits                          Waiting Audits
GET   /audits/{audit_id}               Read Audit
POST  /audits/{audit_id}/review        Review Audit
GET   /log                             Read Log


Starting an audit returns a **stream**: FastAPI sends each step as a server-sent event, a
`data:` line of JSON and a blank line, as soon as the step ends. The screen shows them
live. Here the app runs in memory, without a server, through httpx:

In [17]:
async def api_demo(service):
    app.state.service = service
    transport = httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transport, base_url="http://api") as client:
        stream = await client.post("/listings/P0171/audits")
        print(stream.headers["content-type"])
        events = stream.text.strip().split("\n\n")
        print(events[0])
        print("...")
        print(events[-1])
        audit_id = json.loads(events[-1].removeprefix("data: "))["audit_id"]

        undecided = await client.post(f"/audits/{audit_id}/review", json={"reviewer": "A"})
        print()
        print(undecided.status_code, undecided.json())
        decision = {"rule": "prior_price", "approved": True}
        review = {"reviewer": "Test Reviewer", "decisions": [decision]}
        saved = await client.post(f"/audits/{audit_id}/review", json=review)
        print(saved.status_code, saved.json()["status"], "|", saved.json()["message"]["subject"])


await service_run(api_demo, ScriptedModel(script=p0171()))

text/event-stream; charset=utf-8
data: {"audit_id":"f8fb410f-da2d-4711-868e-c23a861d5a54","step":"read_listing","detail":"Amazon Brand – Stone & Beam Classic Outdoor Patterened Throw Pillow - ","seconds":0.1}
...
data: {"audit_id":"f8fb410f-da2d-4711-868e-c23a861d5a54","step":"waiting","detail":"One possible problem: the -42% badge.","seconds":0.3}

422 {'detail': ['Approve or reject the finding on Honest discounts (the 30-day rule).']}
200 reviewed | Your listing P0171: 1 possible problem to look at


An incomplete review gets status 422 with the sentences human_review wrote, and the audit
keeps waiting. A request for a listing that doesn't exist gets 404 before any stream
starts: a stream begins with status 200, so after that a failure can only be told inside
it, as an `error` event.

## 6. Who may do what

The design keeps every decision with a person:

| Who | May | May not |
| --- | --- | --- |
| The model | judge the kind, search the law, word findings | change a check's result, flag a listing, write to the database |
| Code | run checks, build the report and the message draft | approve or reject a finding |
| The reviewer | approve, reject, reword a fix | edit the audit log |
| Nobody | | send anything to a seller: the message is a draft |

## 7. The review screen

The screen is a Streamlit app, a Python script that Streamlit runs again from the top on
every click; `st.session_state` keeps what has to last between runs, such as the
reviewer's name. Start the API, then the screen, in two terminals:

```
uv run uvicorn listingaudit.review.api:app
uv run streamlit run src/listingaudit/review/screen.py
```

It opens at http://localhost:8501: the audits waiting on the left, and for the open one
the listing and its photo beside the report, with a decision, a reason and the fix for
each finding. The decisions sit in a form, so the screen sends them in one go when you
save. Seller text is shown with `st.text`, never as Markdown, so it appears exactly as
the seller wrote it. The tests drive the screen with Streamlit's test harness,
`AppTest`, against a stand-in for the API.

To see a paused review survive a restart: audit a listing, stop the API with Ctrl+C,
start it again, and reload the screen. The audit is still waiting.

In [18]:
with psycopg.connect(settings.database_url, autocommit=True) as conn:
    conn.execute(sql.SQL("DROP SCHEMA {} CASCADE").format(sql.Identifier(SCHEMA)))
print(f"Dropped the notebook's schema, {SCHEMA}.")

Dropped the notebook's schema, notebook_review.
